# Ejercicio 2: comparar con evidencia y elegir

El mismo `viajes_features.csv` y la misma preparación importada. Mismo esqueleto del Ejercicio 1, con los cinco modelos de la Semana 10 en un experimento propio. Al final eliges el mejor desde el registro, no desde tu memoria, y compruebas que el modelo guardado reproduce su número.

Los pasos 1 a 3 tienen `TODO`; los pasos 4 y 5 solo dicen qué variable dejar. La última celda imprime una comprobación por paso; no la modifiques. Corre todas las celdas una vez antes de empezar (**Run All**). Si vuelves a correr las celdas que entrenan, aparecen corridas repetidas en el registro: es lo esperado.

In [ ]:
import hashlib
import logging
import time

import mlflow
import mlflow.sklearn
from mlflow.entities import Experiment
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from preparacion_taxis import dividir, preparacion

# MLflow avisa en cada corrida que el modelo se guarda sin firma (la
# descripcion de sus columnas de entrada). La firma importa al servir el
# modelo, en la Semana 12; hoy solo ensucia la salida.
logging.getLogger("mlflow").setLevel(logging.ERROR)

viajes = pd.read_csv("viajes_features.csv")
X_ent, X_prueba, y_ent, y_prueba = dividir(viajes)
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)


def huella(ruta):
    with open(ruta, "rb") as archivo:
        return hashlib.md5(archivo.read()).hexdigest()[:12]


huella_datos = huella("viajes_features.csv")
EXPERIMENTO = "tarifa-taxis-comparacion"

## Paso 1: el experimento

In [ ]:
# El registro vive en el servidor de MLflow del Codespace (puerto 5050). La
# direccion viene de la variable de entorno MLFLOW_TRACKING_URI, como en un
# equipo de datos: el codigo no la escribe. Sin esa variable (en tu laptop),
# MLflow guarda el registro en una carpeta mlruns al lado del notebook.
print("Registro de MLflow:", mlflow.get_tracking_uri())

active_experiment_repo: Experiment = mlflow.set_experiment(experiment_name=EXPERIMENTO)
print("| Information Center |".center(65,'='))
print(f"\tNombre registrado del experimento: {active_experiment_repo.name}")
print(
    f"\tFecha de creacion del registro: "
    f"{time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(active_experiment_repo.creation_time / 1000))}"
)
print(f"\tID registrado del experimento: {active_experiment_repo.experiment_id}")
print(f"\tURI de alojamiento de resultados: {active_experiment_repo.artifact_location}")
print("".center(60,'='))

## Paso 2: entrenar y registrar

In [ ]:
def evaluar(nombre, modelo, X_ent, X_prueba, y_ent, y_prueba, semilla_division=42):
    """Entrena el modelo despues de la preparacion, mide el error absoluto
    medio en entrenamiento y en prueba, y registra la corrida en MLflow."""
    pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo)])

    # [DONE]] 2: abre una corrida con mlflow.start_run(run_name=nombre) y
    # registra con mlflow.log_params un diccionario con los hiperparametros
    # del modelo (modelo.get_params()), mas "semilla_division",
    # "huella_datos" y "filas_entrenamiento".

    #? Internally within each of the experiments we can store a bunch of runs, these are individual training
    #? processes that hold information about what went in, into the model, as well as information like
    #? training parameters, data parameters and even results from the model training. In Python this is
    #? all stored within an MLflow ActiveRun object.

    #? An interesting thing higlighted by copliot is that this should not be done throuh a simple start_run, rather
    #? MLFlow is capable of basically following information such as end time and start time as well as closing
    #? the run on its own, so we can simpify this by doing
    with mlflow.start_run(run_name=nombre) as evaluate_run:
        print("| Information Center |".center(65,'='))
        print(f"\tID de la ejecucion: {evaluate_run.info.run_id}")
        print(f"\tNombre de la ejecucion: {evaluate_run.info.run_name}")
        print(f"\tID del experimento: {evaluate_run.info.experiment_id}")
        print(f"\tStart Time de la ejecucion: {evaluate_run.info.start_time}")
        print("".center(60,'='))

        #? Now we register the parameters using the log_params method
        finalized_hyperparameters_to_log = pipeline['modelo'].get_params()
        finalized_hyperparameters_to_log['semilla_division'] = semilla_division
        finalized_hyperparameters_to_log['huella_datos'] = huella_datos
        finalized_hyperparameters_to_log['filas_entrenamiento'] = len(X_ent)
        #? OJO: we can write the run id, although the context manager here already informs that to
        #? any call to log_armaers or any other calls that we do to the direct mlflow object.
        mlflow.log_params(finalized_hyperparameters_to_log,run_id=evaluate_run.info.run_id)

        #? Here we execute the normal training block that came with the function
        inicio = time.perf_counter()
        pipeline.fit(X_ent, y_ent)
        segundos = time.perf_counter() - inicio
        mae_ent = mean_absolute_error(y_ent, pipeline.predict(X_ent))
        mae_prueba = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))

        # [DONE] 3: registra con mlflow.log_metrics "mae_entrenamiento",
        # "mae_prueba" y "segundos".
        finalized_metrics = {
            "mae_entrenamiento": mae_ent,
            "mae_prueba": mae_prueba,
            "segundos": segundos
        }
        mlflow.log_metrics(finalized_metrics, None, run_id=evaluate_run.info.run_id)

        # [DONE] 4: guarda el pipeline entrenado en la corrida con
        mlflow.sklearn.log_model(
            pipeline, #? This is the object that will be serialized into the MLFlow storage system out of this training
            #? step
            "model" #? This is the PATH where the model will be save alongside with requirements.txt, and the
            #? actual serialized model
        )

    #? Since context managers in python don't block access outsie of their scope this will work :0
    print(f"{nombre:20s} division {semilla_division:2d}   entrenamiento {mae_ent:7.4f}   prueba {mae_prueba:7.4f}")

## Paso 3: los cinco modelos de la Semana 10

In [ ]:
# TODO 4: registra los cinco modelos de la Semana 10, con estos nombres:
# "linea base" (DummyRegressor con la mediana), "ridge" (Ridge con
# alpha=10.0), "bosque aleatorio" (200 arboles, random_state=42),
# "gradient boosting" (random_state=42) y "red neuronal" (MLPRegressor con
# una capa de 64 neuronas, max_iter=1000, random_state=42).

evaluar("linea base", DummyRegressor(strategy='median'), X_ent, X_prueba, y_ent, y_prueba, semilla_division=42)
evaluar("ridge", Ridge(alpha=10.0, random_state=42), X_ent, X_prueba, y_ent, y_prueba, semilla_division=42)
evaluar("bosque aleatorio", RandomForestRegressor(n_estimators=200, random_state=42), X_ent, X_prueba, y_ent, y_prueba, semilla_division=42)
evaluar("gradient boosting", GradientBoostingRegressor(random_state=42), X_ent, X_prueba, y_ent, y_prueba,  semilla_division=42)
evaluar("red neuronal", MLPRegressor(hidden_layer_sizes=(64,), max_iter=1000, random_state=42), X_ent, X_prueba, y_ent, y_prueba, semilla_division=42)

## Paso 4: el registro, ordenado

In [ ]:
# Deja en corridas las corridas del experimento (mlflow.search_runs),
# ordenadas de menor a mayor mae_prueba, y en mejor_run_id el run_id de la
# primera.

#? What we are going to do here is effectivly look up the runs taht we arleady did and store them, the object
#? that we are working with works like this. IN theory since we are performing individualized search, each of
#? these is going to return an expanded pandas DataFrame which we are going to have to collapse on each
#? other into a single dataFrame and then we will lookup the run_id of the lowest eval MAE

#? Upon initial assessment  the resulting dataframe that comes from this stores ALL of the runs directly. I thought we would have to filter by name , and while we know them by name this is an interesting approach, we can in theory tell it to order by a column, and since we know that each of these expands in the form of
#? metrics.* and that star is our local name we can definitely say metrics.mae_prueba as requested.
print(mlflow.search_runs(experiment_names=[EXPERIMENTO]))

In [ ]:
corridas: pd.DataFrame = mlflow.search_runs(experiment_names=[EXPERIMENTO], order_by=["metrics.mae_prueba ASC"])
corridas

In [ ]:
#? Now we know taht the resulting DataFrame is oredered by mae prueba in an ASCENDING format, i.e. lowest or best mae should be index 1. We can use .iloc to grab the row by index and then extract its run_id
#? directly through single column access
mejor_run_id = corridas.iloc[0]['run_id']
mejor_run_name = corridas.iloc[0]['tags.mlflow.runName']
print("| Information Center |".center(65,'='))
print(f"\tMejor MAE Run ID: {mejor_run_id}")
print(f"\tMejor MAE Run Name: {mejor_run_name}")
print("".center(60,'='))

## Paso 5: marcar el ganador y reproducirlo

In [ ]:
# Marca la corrida ganadora con la etiqueta "candidato" = "final"
# (mlflow.MlflowClient().set_tag). Despues carga su modelo desde el
# registro con mlflow.sklearn.load_model("runs:/" + mejor_run_id + "/modelo"),
# calcula su error absoluto medio sobre X_prueba y dejalo en
# mae_reproducido.

#? Here we have two approaches, first we know we need to add a single tag to the best model, and we know
#? its ID, we can use that to filter and find the direct object.
mlflow.MlflowClient().set_tag( run_id=mejor_run_id, key='candidato',value='final')

#? Having done that the corresponding tag has been applied. Now to load it again we are going to use the
#? previous id. We know in the table that we gathered an artifact  uri which should lead us directly to the location where the model is stored . The recommended approach by copilot and MLFLow is to use the runs:/
#? approach, but I want to let it be clear that the artifact URI would drop us exactly at the URI of the root folder for the artifacts of taht run.

loaded_model_from_runs = mlflow.sklearn.load_model("runs:/" + mejor_run_id + "/model")
if loaded_model_from_runs:
    print("Modelo cargado correctamente")
else:
    print("Modelo no cargado")

#? Now we reproduce the mae through the direct function
mae_reproducido = mean_absolute_error(y_prueba, loaded_model_from_runs.predict(X_prueba))
print(f"MAE Reproducido del modelo cargado: {round(mae_reproducido,4)}")

## Comprobación

In [ ]:
def comprobar(numero, condicion):
    if condicion:
        print(f"Paso {numero}: OK")
    else:
        print(f"Paso {numero}: revisa")


experimento = mlflow.get_experiment_by_name(EXPERIMENTO)
registradas = None
if experimento is not None:
    registradas = mlflow.search_runs(experiment_ids=[experimento.experiment_id])
nombres = set()
if registradas is not None and len(registradas) > 0:
    nombres = set(registradas["tags.mlflow.runName"])

comprobar(1, experimento is not None)
comprobar(2, registradas is not None and "params.huella_datos" in registradas and (registradas["params.huella_datos"] == huella_datos).all())
comprobar(3, {"linea base", "ridge", "bosque aleatorio", "gradient boosting", "red neuronal"} <= nombres and registradas["metrics.mae_prueba"].notna().all())
comprobar(4, corridas is not None and mejor_run_id == corridas.loc[0, "run_id"] and corridas.loc[0, "tags.mlflow.runName"] == "red neuronal")
etiqueta = None
if mejor_run_id is not None:
    etiqueta = mlflow.MlflowClient().get_run(mejor_run_id).data.tags.get("candidato")
comprobar(5, etiqueta == "final" and mae_reproducido is not None and abs(mae_reproducido - corridas.loc[0, "metrics.mae_prueba"]) < 1e-9)

## Si terminas antes

Abre la interfaz de MLflow (pestaña **Ports** de VS Code, puerto 5050, ícono del globo), entra al experimento `tarifa-taxis-comparacion`, selecciona las cinco corridas y compáralas con **Compare**. Además del error, ¿qué te dice el registro de la red neuronal que no te diría una tabla escrita a mano?